# FenriX — Identification Benchmark (collect & grade)

Runs each local model over the 8 anonymized companies and records its **company guess + reasoning + tokens + latency**.

**Data:** the anonymized real-filing corpus in `../data/anonymized_filings_7yr/` (8 companies as numbered dirs `1`-`8` of raw SEC full-submission dumps). `extract_reports.py` pulls each company's most-recent 10-K Business section into `../data/processed_reports/company_N.txt` — that extracted text is what gets fed to the models.

**How it works:** for each `(model, thinking-mode)` it launches `llama-server` (native GPU) via `../models/scripts/serve.sh`, sends each report through the identification prompt, saves raw answers to `../models/results/identifications_v2/<model>__<mode>.json`, then consolidates into `_grading.csv`.

**Prereqs:** models downloaded (`../models/scripts/download.sh`) and native llama.cpp built (`~/.local/bin/llama-server`).


In [ ]:
# Ensure deps (safe to re-run)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "pyyaml", "tqdm"])
print("deps ok")


In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
import os, json, time, subprocess, urllib.request
from pathlib import Path
import yaml

NB_DIR      = Path.cwd()                                  # notebooks/
MODELS_DIR  = (NB_DIR / ".." / "models").resolve()
RAW_DIR     = (NB_DIR / ".." / "data" / "anonymized_filings_7yr").resolve()   # raw SEC dumps, dirs 1-8
REPORTS_DIR = (NB_DIR / ".." / "data" / "processed_reports").resolve()        # extracted Business sections
OUT_DIR     = MODELS_DIR / "results" / "identifications_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

PORT = 8080
CTX  = 16384          # extracted reports are ~9k words (~12k tokens); 8192 is too small

# Models to run (registry ids). llama33-70b is offload-slow (~3 tok/s) -> skipped by default.
REG        = yaml.safe_load(open(MODELS_DIR / "registry.yaml"))
ALL_MODELS = [m["id"] for m in REG["models"]]
SKIP       = {"llama33-70b"}                             # remove to include the slow 70B
MODELS     = [m for m in ALL_MODELS if m not in SKIP]

# Thinking dimension. "nothink" = enable_thinking False (crisp). "think" = reasons first.
THINKING_MODES = ["nothink"]                             # add "think" to test if reasoning helps ID
MAX_TOKENS     = {"nothink": 700, "think": 4000}

# Ground-truth answer key (dir N -> real company), from the anonymization project's entities.yaml.
GROUND_TRUTH = {1: "AMD", 2: "AppLovin", 3: "General Motors", 4: "Kroger",
                5: "McKesson", 6: "Netflix", 7: "Palantir", 8: "SanDisk"}
COMPANIES = sorted(GROUND_TRUTH)                          # [1, 2, ..., 8]

PROMPT_TMPL = (
    "You are given an anonymized annual report (Form 10-K). All company, product, "
    "person, and place names have been replaced with fictional ones, and dollar "
    "figures are illustrative. Based on the business description, sector, product/"
    "segment lineup, competitive positioning, and financial profile, identify the "
    "REAL publicly-traded company this report is most likely modeled on.\n\n"
    "Answer with:\n1) Your single best guess (one real company name).\n"
    "2) 3-5 specific clues from the text that led you there.\n\n"
    "=== REPORT ===\n{report}"
)

print("models   :", MODELS)
print("modes    :", THINKING_MODES)
print("companies:", COMPANIES)


In [ ]:
# ── Extract Business narratives from the raw anonymized filings ──────────────
# data/anonymized_filings_7yr/<N>/*.txt  ->  data/processed_reports/company_N.txt
# (skips if already extracted; delete data/processed_reports/ to force a re-run)
if not list(REPORTS_DIR.glob("company_*.txt")):
    subprocess.run([__import__("sys").executable, str(NB_DIR / "extract_reports.py")], check=True)
print("extracted reports:", sorted(p.name for p in REPORTS_DIR.glob("company_*.txt")))


In [ ]:
# ── Helpers: server lifecycle + one identification call ──────────────────────
LLAMA_ENV = {**os.environ, "PATH": f"{Path.home()}/.local/bin:" + os.environ.get("PATH", "")}

def start_server(model_id):
    """serve.sh exec's llama-server (native GPU). Thinking is toggled per-request
    via enable_thinking, so ONE server per model handles both modes."""
    args = ["bash", str(MODELS_DIR / "scripts" / "serve.sh"), model_id, "--ctx-size", str(CTX)]
    log = open(OUT_DIR / f"_server_{model_id}.log", "w")
    proc = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT,
                            env=LLAMA_ENV, cwd=str(MODELS_DIR))
    return proc, log

def wait_ready(port, timeout=240):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=3) as r:
                if r.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(2)
    return False

def stop_server(proc, log):
    proc.terminate()
    try:    proc.wait(timeout=20)
    except Exception: proc.kill()
    log.close()

def identify(port, report_path, mode):
    report = Path(report_path).read_text(encoding="utf-8", errors="replace")
    body = json.dumps({
        "messages": [{"role": "user", "content": PROMPT_TMPL.format(report=report)}],
        "temperature": 0.3, "max_tokens": MAX_TOKENS[mode],
        # per-request thinking toggle -- works for both Gemma 4 and Qwen3.5
        "chat_template_kwargs": {"enable_thinking": mode == "think"},
    }).encode()
    req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 data=body, headers={"Content-Type": "application/json"})
    t0 = time.time()
    resp = json.load(urllib.request.urlopen(req, timeout=900))
    dt = time.time() - t0
    ch = resp["choices"][0]; m = ch.get("message", {}); u = resp.get("usage", {})
    return {
        "guess":             (m.get("content") or "").strip(),
        "reasoning":         (m.get("reasoning_content") or "").strip(),
        "finish_reason":     ch.get("finish_reason"),
        "prompt_tokens":     u.get("prompt_tokens"),
        "completion_tokens": u.get("completion_tokens"),
        "latency_s":         round(dt, 1),
    }

print("helpers ready")


In [ ]:
# ── Run: per model -> serve once -> all (modes x companies) -> save json ─────
from tqdm.notebook import tqdm

def run_model(model_id):
    proc, log = start_server(model_id)
    try:
        if not wait_ready(PORT):
            print(f"  server not ready for {model_id} - skipping (see {log.name})")
            return
        for mode in THINKING_MODES:
            rows = []
            for comp in tqdm(COMPANIES, desc=f"{model_id}/{mode}", leave=False):
                rp = REPORTS_DIR / f"company_{comp}.txt"
                try:
                    r = identify(PORT, rp, mode)
                except Exception as e:
                    r = {"guess": "", "reasoning": "", "error": str(e)}
                r.update({"model": model_id, "mode": mode, "company": comp})
                rows.append(r)
            json.dump(rows, open(OUT_DIR / f"{model_id}__{mode}.json", "w"), indent=2)
            ok = sum(1 for r in rows if r.get("guess"))
            print(f"  {model_id}/{mode}: {ok}/{len(rows)} answered")
    finally:
        stop_server(proc, log)

for model_id in MODELS:
    print(f"> {model_id}")
    run_model(model_id)
print("done.")


In [ ]:
# ── Consolidate every identification into one grading table ──────────────────
import pandas as pd, glob

recs = []
for f in sorted(glob.glob(str(OUT_DIR / "*__*.json"))):
    recs.extend(json.load(open(f)))

df = pd.DataFrame(recs)
if df.empty:
    print("no identifications yet - run the cell above first")
else:
    df["real"] = df["company"].map(GROUND_TRUTH)          # answer key (dir N -> real company)
    cols = ["model", "mode", "company", "real", "guess", "reasoning",
            "completion_tokens", "latency_s", "finish_reason"]
    df = df[[c for c in cols if c in df.columns]]
    grade = df.copy()
    grade["correct"] = ""   # you fill: 1 / 0  (compare guess vs. the 'real' column)
    grade["notes"]   = ""
    grade.to_csv(OUT_DIR / "_grading.csv", index=False)
    print(f"wrote {OUT_DIR/'_grading.csv'}  ({len(grade)} rows)")

    def short(s, n=90):
        s = (s or "").replace("\n", " ")
        return s[:n] + ("..." if len(s) > n else "")
    view = df.copy(); view["guess"] = view["guess"].map(short)
    display(view[["model", "mode", "company", "real", "guess", "completion_tokens", "latency_s"]])


## Grading

Open `../models/results/identifications_v2/_grading.csv`. The **`real`** column is the answer key (dir `N` -> real company) — fill `correct` (1/0) by comparing each `guess` to it. Each `<model>__<mode>.json` holds the full guess + reasoning.

**Answer key:** `1`=AMD `2`=AppLovin `3`=GM `4`=Kroger `5`=McKesson `6`=Netflix `7`=Palantir `8`=SanDisk

**Dimensions to compare:**
- **provider** — Gemma 4 vs Qwen3.5 vs Llama
- **size tier** — big-MoE vs mid-dense vs small-fast
- **thinking** — add `"think"` to `THINKING_MODES` and re-run to test whether reasoning improves identification (costs many more tokens)

Cross-reference speed from `../models/results/_summary.txt`.

> ⚠️ The corpus is only *name*-anonymized — real third-party entities/events leak (e.g. Palantir's report still names "Airbus"/"Skywise"), so identification is easier than a true de-identification would allow. Treat accuracy as an upper bound.
